# AWS Resource Cleanup — BINFX410 Chapter 10

Run this notebook **after completing all exercises** to delete every AWS resource created during the module and avoid ongoing charges.

### What gets deleted

| Step | Resource | From exercise |
|------|----------|--------------|
| 1 | Kinesis Firehose delivery stream | Ex 9 |
| 2 | Lake Formation permissions, data-cell filters, registered S3 location | Ex 8 |
| 3 | IAM roles (firehose, researcher, clinical) | Ex 8, 9 |
| 4 | Glue crawlers | Ex 3 |
| 5 | Glue databases + all tables (bronze, silver, gold, sra, streaming, mart) | Ex 1–10 |
| 6 | All objects in both S3 buckets | Ex 1–10 |
| 7 | The S3 buckets themselves | Ex 1–10 |

> **Safe to re-run.** Every step catches `ResourceNotFound`-style errors so you can run this notebook even if you only completed a subset of exercises.

---
**Set `STUDENT_ID` in the Setup cell before running.**

In [ ]:
import boto3
from botocore.exceptions import ClientError

# ── CONFIGURE THIS ────────────────────────────────────────────────────────────
STUDENT_ID = "sab032226"   # e.g. "sab032226"
AWS_REGION = "us-east-1"
# ──────────────────────────────────────────────────────────────────────────────

BUCKET       = f"binfx410-datalake-{STUDENT_ID}"
ATHENA_BUCKET = f"binfx410-athena-results-{STUDENT_ID}"

s3       = boto3.client('s3',            region_name=AWS_REGION)
glue     = boto3.client('glue',          region_name=AWS_REGION)
iam      = boto3.client('iam',           region_name=AWS_REGION)
firehose = boto3.client('firehose',      region_name=AWS_REGION)
lf       = boto3.client('lakeformation', region_name=AWS_REGION)

# ── Resource manifests ────────────────────────────────────────────────────────
GLUE_DATABASES = [
    'binfx410_bronze',
    'binfx410_silver',
    'binfx410_gold',
    'binfx410_sra',
    'binfx410_streaming',
    'binfx410_mart',
]

GLUE_CRAWLERS = [
    f'binfx410-bronze-crawler-{STUDENT_ID}',
]

FIREHOSE_STREAMS = [
    f'binfx410-sequencer-feed-{STUDENT_ID}',
]

IAM_ROLES = [
    f'binfx410-firehose-role-{STUDENT_ID}',
    'binfx410-researcher-role',
    'binfx410-clinical-role',
]

S3_BUCKETS = [BUCKET, ATHENA_BUCKET]

print(f"Student ID : {STUDENT_ID}")
print(f"Region     : {AWS_REGION}")
print(f"Data bucket: s3://{BUCKET}")
print(f"Athena bucket: s3://{ATHENA_BUCKET}")

In [ ]:
# ── Preview: show which resources actually exist before deleting ───────────────

def check(label, exists_fn):
    try:
        exists_fn()
        print(f"  FOUND    {label}")
        return True
    except ClientError as e:
        code = e.response['Error']['Code']
        print(f"  missing  {label}  ({code})")
        return False

print("=== S3 Buckets ===")
for b in S3_BUCKETS:
    try:
        s3.head_bucket(Bucket=b)
        resp  = s3.list_objects_v2(Bucket=b)
        print(f"  FOUND    s3://{b}  ({resp.get('KeyCount', 0)} objects visible)")
    except ClientError as e:
        print(f"  missing  s3://{b}  ({e.response['Error']['Code']})")

print("\n=== Glue Databases ===")
for db in GLUE_DATABASES:
    try:
        glue.get_database(Name=db)
        tables = glue.get_tables(DatabaseName=db).get('TableList', [])
        print(f"  FOUND    {db}  ({len(tables)} tables)")
    except ClientError:
        print(f"  missing  {db}")

print("\n=== Glue Crawlers ===")
for c in GLUE_CRAWLERS:
    check(c, lambda c=c: glue.get_crawler(Name=c))

print("\n=== Kinesis Firehose Streams ===")
for stream in FIREHOSE_STREAMS:
    check(stream, lambda s=stream: firehose.describe_delivery_stream(DeliveryStreamName=s))

print("\n=== IAM Roles ===")
for role in IAM_ROLES:
    check(role, lambda r=role: iam.get_role(RoleName=r))

print("\n=== Lake Formation Registered Resources ===")
try:
    resources = lf.list_resources().get('ResourceInfoList', [])
    student_resources = [r for r in resources if STUDENT_ID in r.get('ResourceArn', '')]
    if student_resources:
        for r in student_resources:
            print(f"  FOUND    {r['ResourceArn']}")
    else:
        print("  (none matching student ID)")
except ClientError as e:
    print(f"  (could not list: {e.response['Error']['Code']})")

In [ ]:
# ── Step 1: Kinesis Firehose delivery streams (Exercise 9) ────────────────────
import time

print("Step 1: Deleting Kinesis Firehose streams...")

for stream_name in FIREHOSE_STREAMS:
    try:
        # Check current state — must be ACTIVE before we can delete
        desc = firehose.describe_delivery_stream(DeliveryStreamName=stream_name)
        state = desc['DeliveryStreamDescription']['DeliveryStreamStatus']
        
        if state == 'CREATING':
            print(f"  Waiting for {stream_name} to finish creating...")
            for _ in range(12):  # up to ~60 seconds
                time.sleep(5)
                desc = firehose.describe_delivery_stream(DeliveryStreamName=stream_name)
                if desc['DeliveryStreamDescription']['DeliveryStreamStatus'] == 'ACTIVE':
                    break
        
        firehose.delete_delivery_stream(DeliveryStreamName=stream_name)
        print(f"  Deleted: {stream_name}")
    except ClientError as e:
        code = e.response['Error']['Code']
        if code == 'ResourceNotFoundException':
            print(f"  Skipped: {stream_name} (not found)")
        else:
            print(f"  Error:   {stream_name} — {code}: {e.response['Error']['Message']}")

print("Done.")

In [ ]:
# ── Step 2: Lake Formation cleanup (Exercise 8) ───────────────────────────────
# Order: revoke permissions → delete data-cell filters → deregister S3 resource

print("Step 2: Cleaning up Lake Formation...")

# 2a. Revoke all LF permissions for student-created IAM roles
print("  Revoking LF permissions for exercise roles...")
for role_name in IAM_ROLES:
    try:
        role_arn = iam.get_role(RoleName=role_name)['Role']['Arn']
        perms = lf.list_permissions(
            Principal={'DataLakePrincipalIdentifier': role_arn}
        ).get('PrincipalResourcePermissions', [])
        
        for perm in perms:
            try:
                lf.revoke_permissions(
                    Principal=perm['Principal'],
                    Resource=perm['Resource'],
                    Permissions=perm['Permissions'],
                    PermissionsWithGrantOption=perm.get('PermissionsWithGrantOption', [])
                )
            except ClientError:
                pass  # already revoked or resource gone
        
        if perms:
            print(f"    Revoked {len(perms)} permission(s) for {role_name}")
    except ClientError:
        pass  # role doesn't exist or LF not enabled

# 2b. Delete data-cell filters created in Exercise 8
print("  Deleting data-cell filters...")
CELL_FILTERS = [
    {'TableCatalogId': boto3.client('sts').get_caller_identity()['Account'],
     'DatabaseName': 'binfx410_silver',
     'TableName': 'patient_genomics',
     'Name': 'brain_tissue_only'},
]
for cf in CELL_FILTERS:
    try:
        lf.delete_data_cells_filter(**cf)
        print(f"    Deleted filter: {cf['Name']}")
    except ClientError as e:
        code = e.response['Error']['Code']
        if code in ('EntityNotFoundException', 'AccessDeniedException'):
            print(f"    Skipped filter: {cf['Name']} ({code})")
        else:
            print(f"    Error:   {cf['Name']} — {code}")

# 2c. Deregister the S3 data lake resource
print("  Deregistering S3 data lake location...")
bucket_arn = f"arn:aws:s3:::{BUCKET}"
try:
    lf.deregister_resource(ResourceArn=bucket_arn)
    print(f"    Deregistered: {bucket_arn}")
except ClientError as e:
    code = e.response['Error']['Code']
    if code in ('EntityNotFoundException', 'AccessDeniedException'):
        print(f"    Skipped: {bucket_arn} ({code})")
    else:
        print(f"    Error:   {code}: {e.response['Error']['Message']}")

print("Done.")

In [ ]:
# ── Step 3: IAM roles (Exercises 8, 9) ────────────────────────────────────────
# Must detach managed policies and delete inline policies before deleting role.

print("Step 3: Deleting IAM roles...")

def delete_iam_role(role_name):
    try:
        iam.get_role(RoleName=role_name)
    except ClientError:
        print(f"  Skipped: {role_name} (not found)")
        return

    # Remove from any instance profiles
    for profile in iam.list_instance_profiles_for_role(RoleName=role_name).get('InstanceProfiles', []):
        iam.remove_role_from_instance_profile(
            InstanceProfileName=profile['InstanceProfileName'],
            RoleName=role_name
        )

    # Detach managed policies
    for policy in iam.list_attached_role_policies(RoleName=role_name).get('AttachedPolicies', []):
        iam.detach_role_policy(RoleName=role_name, PolicyArn=policy['PolicyArn'])

    # Delete inline policies
    for policy_name in iam.list_role_policies(RoleName=role_name).get('PolicyNames', []):
        iam.delete_role_policy(RoleName=role_name, PolicyName=policy_name)

    iam.delete_role(RoleName=role_name)
    print(f"  Deleted: {role_name}")

for role in IAM_ROLES:
    try:
        delete_iam_role(role)
    except ClientError as e:
        print(f"  Error:   {role} — {e.response['Error']['Code']}: {e.response['Error']['Message']}")

print("Done.")

In [ ]:
# ── Step 4: Glue crawlers (Exercise 3) ───────────────────────────────────────

print("Step 4: Deleting Glue crawlers...")

for crawler_name in GLUE_CRAWLERS:
    try:
        # Stop the crawler first if it's running
        info = glue.get_crawler(Name=crawler_name)
        state = info['Crawler']['State']
        if state == 'RUNNING':
            glue.stop_crawler(Name=crawler_name)
            print(f"  Stopped running crawler: {crawler_name}")
        glue.delete_crawler(Name=crawler_name)
        print(f"  Deleted: {crawler_name}")
    except ClientError as e:
        code = e.response['Error']['Code']
        if code == 'EntityNotFoundException':
            print(f"  Skipped: {crawler_name} (not found)")
        else:
            print(f"  Error:   {crawler_name} — {code}: {e.response['Error']['Message']}")

print("Done.")

In [ ]:
# ── Step 5: Glue databases (Exercises 1–10) ───────────────────────────────────
# Deleting a database cascades to all its tables — no need to delete tables individually.

print("Step 5: Deleting Glue databases (tables cascade)...")

for db_name in GLUE_DATABASES:
    try:
        tables = glue.get_tables(DatabaseName=db_name).get('TableList', [])
        glue.delete_database(Name=db_name)
        print(f"  Deleted: {db_name}  ({len(tables)} tables removed)")
    except ClientError as e:
        code = e.response['Error']['Code']
        if code == 'EntityNotFoundException':
            print(f"  Skipped: {db_name} (not found)")
        else:
            print(f"  Error:   {db_name} — {code}: {e.response['Error']['Message']}")

print("Done.")

In [ ]:
# ── Step 6: Empty both S3 buckets ─────────────────────────────────────────────
# Handles both non-versioned and versioned objects (delete markers included).

print("Step 6: Emptying S3 buckets...")

def empty_bucket(bucket_name):
    deleted_count = 0

    # Delete current objects (non-versioned path)
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket_name):
        objects = [{'Key': obj['Key']} for obj in page.get('Contents', [])]
        if objects:
            s3.delete_objects(Bucket=bucket_name, Delete={'Objects': objects})
            deleted_count += len(objects)

    # Delete all versions + delete markers (versioned bucket path)
    ver_paginator = s3.get_paginator('list_object_versions')
    for page in ver_paginator.paginate(Bucket=bucket_name):
        items = [
            {'Key': v['Key'], 'VersionId': v['VersionId']}
            for v in page.get('Versions', []) + page.get('DeleteMarkers', [])
        ]
        if items:
            s3.delete_objects(Bucket=bucket_name, Delete={'Objects': items})
            deleted_count += len(items)

    print(f"  Emptied: s3://{bucket_name}  ({deleted_count} objects deleted)")

for bucket in S3_BUCKETS:
    try:
        s3.head_bucket(Bucket=bucket)
        empty_bucket(bucket)
    except ClientError as e:
        code = e.response['Error']['Code']
        if code in ('404', 'NoSuchBucket'):
            print(f"  Skipped: s3://{bucket} (not found)")
        else:
            print(f"  Error:   s3://{bucket} — {code}: {e.response['Error']['Message']}")

print("Done.")

In [ ]:
# ── Step 7: Delete the S3 buckets ─────────────────────────────────────────────
# Buckets must be empty before they can be deleted (previous step handles this).

print("Step 7: Deleting S3 buckets...")

for bucket in S3_BUCKETS:
    try:
        s3.delete_bucket(Bucket=bucket)
        print(f"  Deleted: s3://{bucket}")
    except ClientError as e:
        code = e.response['Error']['Code']
        if code == 'NoSuchBucket':
            print(f"  Skipped: s3://{bucket} (not found)")
        elif code == 'BucketNotEmpty':
            print(f"  Error:   s3://{bucket} is not empty — re-run Step 6")
        else:
            print(f"  Error:   s3://{bucket} — {code}: {e.response['Error']['Message']}")

print("Done.")

In [ ]:
# ── Verification: confirm nothing remains ─────────────────────────────────────

print("=== Verification ===")
all_clean = True

print("\nS3 Buckets:")
for b in S3_BUCKETS:
    try:
        s3.head_bucket(Bucket=b)
        print(f"  STILL EXISTS  s3://{b}  ← needs attention")
        all_clean = False
    except ClientError:
        print(f"  gone          s3://{b}")

print("\nGlue Databases:")
for db in GLUE_DATABASES:
    try:
        glue.get_database(Name=db)
        print(f"  STILL EXISTS  {db}  ← needs attention")
        all_clean = False
    except ClientError:
        print(f"  gone          {db}")

print("\nKinesis Firehose:")
for stream in FIREHOSE_STREAMS:
    try:
        firehose.describe_delivery_stream(DeliveryStreamName=stream)
        print(f"  STILL EXISTS  {stream}  ← needs attention")
        all_clean = False
    except ClientError:
        print(f"  gone          {stream}")

print("\nIAM Roles:")
for role in IAM_ROLES:
    try:
        iam.get_role(RoleName=role)
        print(f"  STILL EXISTS  {role}  ← needs attention")
        all_clean = False
    except ClientError:
        print(f"  gone          {role}")

print()
if all_clean:
    print("All resources deleted. No further AWS charges will accrue for this module.")
else:
    print("Some resources still exist — see items marked above and re-run the relevant step.")